In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/NBA_PlayerGameLogs23_24.csv")



| Feature                | Why it might help predict points                          |
| ---------------------- | --------------------------------------------------------- |
| Previous game points   | Captures immediate form                                   |
| Rolling 5-game average | Smooths out random game-to-game noise                     |
| Days of rest           | Fatigue and recovery can affect performance               |
| Home/Away              | Home-court advantage may influence performance            |
| Opponent               | Different defenses present different levels of difficulty |




In [4]:
#Cast the GAME_DATE column to the datetime type and then sort by both it and PLAYER_ID for grouping and to establish a chronological order.
df['GAME_DATE'] =  pd.to_datetime(df['GAME_DATE']) 

df = df.sort_values(['PLAYER_ID', 'GAME_DATE' ,'GAME_ID'])

Rolling Statistics

In [ ]:
#Create rolling statistics


#Create column for a player's points from their previous game

df['PREV_GAME_PTS'] = df.groupby('PLAYER_ID')['PTS'].shift(1)
df["PREV_GAME_PTS"] = df["PREV_GAME_PTS"].fillna(0)


df['AVG_PTS_LAG5'] = df.groupby('PLAYER_ID')['PTS'].rolling(5).mean().reset_index(level=0,drop=True)
df = df.dropna(subset= ['AVG_PTS_LAG5'])





Home/Away- Apparently, in the matchup column, if it says "vs" in the string, it's a home game for the team before it and it's an away game if it contains "@"

In [ ]:
df['HOME/AWAY'] =  pd.Categorical(np.where(df['MATCHUP'].str.contains('@'), 'Away', 'Home'))


Rest Days

In [ ]:
df["REST_DAYS"] = (
    df.groupby("PLAYER_ID")["GAME_DATE"]
      .diff()
      .dt.days
      .fillna(0)
      .astype(int)
)

df

Opponent

In [ ]:
df['OPPONENT'] = df['MATCHUP'].str[-3:]
df['OPPONENT'] = pd.Categorical(df['OPPONENT'])

df

Export to csv

In [19]:
df.to_csv(
   "../data/preprocessed.csv",
    index=False
)
